In [1]:
import sys
import os

In [2]:
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/nlp'):
        !git clone https://github.com/jaYulichka46/nlp.git
    
    %cd /content/nlp
    !pip install regex pandas stanza scikit-learn -q
    sys.path.append('/content/nlp')

    FOLDER_ID = '1Xhu4xjZpRu-RP730-hyErp5F0C3l_EvO'
    
    os.makedirs('data', exist_ok=True)
    !gdown --folder https://drive.google.com/drive/folders/{FOLDER_ID} -O data/
    
    data_dir = 'data'
else:
    sys.path.append(os.path.abspath('..'))
    data_dir = '../data'

In [3]:
import json
import pandas as pd
from src.ie_rules import extract_all

In [4]:
gold_path = os.path.join(data_dir, "sample", "lab4_gold_ie.jsonl")

metrics = {
    "MONEY_AMOUNT": {"TP": 0, "FP": 0},
    "DATE_EVENT": {"TP": 0, "FP": 0},
    "UA_AGENCY": {"TP": 0, "FP": 0}
}

false_positives = []

In [5]:
if os.path.exists(gold_path):
    with open(gold_path, 'r', encoding='utf-8') as f:
        for line in f:
            case = json.loads(line)
            text = case["text"]
            gold_entities = case["gold"]
            predicted_entities = extract_all(text)
            pred_flat = [{"type": p["field_type"], "value": p["value"], "raw": p} for p in predicted_entities]
            gold_flat = [{"type": g["type"], "value": g["value"]} for g in gold_entities]
            
            for pred in pred_flat:
                match = None
                for g in gold_flat:
                    if g["type"] == pred["type"]:
                        if g["type"] == "MONEY_AMOUNT":
                            if g["value"]["amount"] == pred["value"]["amount"] and g["value"]["currency"] == pred["value"]["currency"]:
                                match = g
                                break
                        elif g["value"] == pred["value"]:
                            match = g
                            break
                
                if match:
                    metrics[pred["type"]]["TP"] += 1
                    gold_flat.remove(match)
                else:
                    metrics[pred["type"]]["FP"] += 1
                    false_positives.append({
                        "text": text,
                        "field_type": pred["type"],
                        "extracted_value": pred["value"]
                    })

    for field, counts in metrics.items():
        tp, fp = counts["TP"], counts["FP"]
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        print(f"{field.ljust(15)}: Precision = {precision:.2f} (TP:{tp}, FP:{fp})")
    print("-"*50)
else:
    print(f"File not found {gold_path}")

MONEY_AMOUNT   : Precision = 1.00 (TP:11, FP:0)
DATE_EVENT     : Precision = 1.00 (TP:10, FP:0)
UA_AGENCY      : Precision = 1.00 (TP:17, FP:0)
--------------------------------------------------


In [6]:
if not false_positives:
    print("All good")
else:
    for i, fp in enumerate(false_positives[:10]):
        print(f"Приклад {i+1}:")
        print(f"Текст: {fp['text']}")
        print(f"витягнуто: {fp['field_type']} -> {fp['extracted_value']}")
        print("-" * 60)

All good


In [7]:
summary_path = "docs/audit_summary_lab4.md" if 'google.colab' in sys.modules else "../docs/audit_summary_lab4.md"

precision_money = metrics["MONEY_AMOUNT"]["TP"] / max((metrics["MONEY_AMOUNT"]["TP"] + metrics["MONEY_AMOUNT"]["FP"]), 1)
precision_date = metrics["DATE_EVENT"]["TP"] / max((metrics["DATE_EVENT"]["TP"] + metrics["DATE_EVENT"]["FP"]), 1)
precision_agency = metrics["UA_AGENCY"]["TP"] / max((metrics["UA_AGENCY"]["TP"] + metrics["UA_AGENCY"]["FP"]), 1)

markdown_content = f"""# Audit Summary Lab 4: Rule-based IE (News Dataset)

## 1. Загальна інформація
* **Домен:** NewsClassificationDataset (Українські новини)
* **Метод:** Rule-based (Regex + Dictionaries)
* **Поля для екстракції:** `MONEY_AMOUNT`, `DATE_EVENT`, `UA_AGENCY`

## 2. Метрики якості (Gold Subset: 30 речень)
Оцінка проводилась із фокусом на **Precision-first** (мінімізація хибних спрацювань).

| Field Type | Precision | True Positives | False Positives |
| :--- | :---: | :---: | :---: |
| **MONEY_AMOUNT** | {precision_money:.2f} | {metrics['MONEY_AMOUNT']['TP']} | {metrics['MONEY_AMOUNT']['FP']} |
| **DATE_EVENT** | {precision_date:.2f} | {metrics['DATE_EVENT']['TP']} | {metrics['DATE_EVENT']['FP']} |
| **UA_AGENCY** | {precision_agency:.2f} | {metrics['UA_AGENCY']['TP']} | {metrics['UA_AGENCY']['FP']} |

## 3. Error Analysis та Edge Cases
* **Абревіатури:** Використання негативних lookahead/lookbehind для кирилиці (`(?<![а-яА-Яіїєґ])`) дозволило знаходити "СБУ" як окреме слово і не витягувати його з усередини інших слів.
* **Гроші:** Алгоритм успішно конвертує "1.5 млрд" у `1500000000.0`, ігноруючи при цьому відсотки чи кількість людей ("10 тисяч мітингувальників").
* **Дати:** Текстові місяці ("15 березня") успішно мапляться на числові значення ("15.03"), а якщо рік відсутній — підставляється поточний.
"""

os.makedirs(os.path.dirname(summary_path), exist_ok=True)
with open(summary_path, "w", encoding="utf-8") as f:
    f.write(markdown_content)

print(f"save to: {summary_path}")

save to: ../docs/audit_summary_lab4.md
